<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/golf/04_normal_discrepancy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Golf putting 4 — Add model discrepancy

Keep the angle-and-distance mean model, but acknowledge that the simple mechanism is not exact. Following the case study, approximate each observed proportion with a Normal distribution and add an error term independent of bin size.

## Setup

This notebook uses the PyMC / ArviZ packages provided by the current Colab environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260923
azp.style.use("arviz-variat")

DATA_BASE = "https://raw.githubusercontent.com/opherdonchin/BayesShortCourse/main/golf/data"
golf = pd.read_csv(f"{DATA_BASE}/broadie_2018_putting.csv")
golf["rate"] = golf["made"] / golf["attempts"]

BALL_RADIUS_FT = (1.68 / 2) / 12
CUP_RADIUS_FT = (4.25 / 2) / 12

golf

In [ ]:
def plot_rate(draws, data, title, *, median_label="median", show_observed=True, ylim=(-0.05, 1.05), show_bounds=False):
    """Plot a probability/rate relationship against continuous putting distance."""
    x = data["distance_ft"].to_numpy()
    curve_dim = next(dim for dim in draws.dims if dim not in ("chain", "draw"))
    order = np.argsort(x)
    draws = draws.isel({curve_dim: order})
    x = x[order]

    median = draws.median(dim=("chain", "draw"))
    interval = draws.azstats.hdi(prob=0.90)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.fill_between(
        x,
        interval.sel(ci_bound="lower"),
        interval.sel(ci_bound="upper"),
        alpha=0.22,
        label="90% HDI",
    )
    ax.plot(x, median, label=median_label)
    if show_observed:
        ax.scatter(x, data["rate"].to_numpy()[order], s=30, color="black", label="observed")
    if show_bounds:
        ax.axhline(0, color="0.6", linewidth=0.8, linestyle="--")
        ax.axhline(1, color="0.6", linewidth=0.8, linestyle="--")
    ax.set(
        xlabel="Distance from hole (feet)",
        ylabel="Proportion made",
        title=title,
    )
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.legend(fontsize=8, frameon=False)
    return ax

def plot_residuals(idata, data, var_name="p_base", title="Residuals"):
    """Plot observed minus posterior-median mechanistic probability."""
    fitted = idata["posterior"][var_name].median(dim=("chain", "draw")).to_numpy()
    x = data["distance_ft"].to_numpy()
    residual = data["rate"].to_numpy() - fitted
    order = np.argsort(x)

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.plot(x[order], residual[order], marker="o")
    ax.set(
        xlabel="Distance from hole (feet)",
        ylabel="Observed − fitted probability",
        title=title,
    )
    return ax

## Model

Let $q_j=y_j/n_j$ be the observed make proportion. We use

$$
q_j\sim N\!\left(p_j,\sqrt{p_j(1-p_j)/n_j+\sigma_{model}^2}\right).
$$

The first variance term is the usual binomial sampling approximation; $\sigma_{model}$ represents mismatch between the simple mechanism and reality.

In [ ]:
coords = {"obs_id": golf["distance_ft"].to_numpy()}
with pm.Model(coords=coords) as model:
    distance = pm.Data("distance", golf["distance_ft"].to_numpy(), dims="obs_id")
    attempts = pm.Data("attempts", golf["attempts"].to_numpy(), dims="obs_id")\n
    sigma_angle_deg = pm.LogNormal("sigma_angle_deg", mu=np.log(2), sigma=0.7)
    sigma_distance = pm.LogNormal("sigma_distance", mu=np.log(0.10), sigma=0.7)
    sigma_model = pm.HalfNormal("sigma_model", sigma=0.01)
    overshot = 1.0
    distance_tolerance = 3.0

    sigma_angle_rad = sigma_angle_deg * np.pi / 180
    threshold_angle = pm.math.arcsin((CUP_RADIUS_FT - BALL_RADIUS_FT) / distance)
    p_angle = pm.Deterministic(
        "p_angle",
        2 * pm.math.invprobit(threshold_angle / sigma_angle_rad) - 1,
        dims="obs_id",
    )
    p_distance = pm.Deterministic(
        "p_distance",
        pm.math.invprobit((distance_tolerance - overshot) / ((distance + overshot) * sigma_distance))
        - pm.math.invprobit(-overshot / ((distance + overshot) * sigma_distance)),
        dims="obs_id",
    )
    p_base = pm.Deterministic("p_base", p_angle * p_distance, dims="obs_id")
    obs_sigma = pm.Deterministic(
        "obs_sigma",
        pm.math.sqrt(p_base * (1 - p_base) / attempts + sigma_model**2),
        dims="obs_id",
    )
    pm.Normal("rate", mu=p_base, sigma=obs_sigma, observed=golf["rate"].to_numpy(), dims="obs_id")

## Prior predictive check

Before fitting, ask what success rates the prior says are plausible across putting distance. The observed rates are shown only as a reference; they do not condition the prior.

The dashed horizontal lines mark the valid probability range. This model can generate values outside $[0,1]$, and the prior predictive display should make that visible rather than clipping it away.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(
        draws=500,
        var_names=["rate"],
        random_seed=RANDOM_SEED,
    )

In [ ]:
plot_rate(
    prior["prior_predictive"]["rate"],
    golf,
    "Prior predictive",
    ylim=None,
    show_bounds=True,
);

## Fit and diagnose

Do not interpret the scientific fit until the sampler diagnostics are acceptable.

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        target_accept=0.92,
        random_seed=RANDOM_SEED,
    )

print("Divergences:", int(idata["sample_stats"]["diverging"].sum().item()))
azs.summary(
    idata,
    var_names=['sigma_angle_deg', 'sigma_distance', 'sigma_model'],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(idata, var_names=['sigma_angle_deg', 'sigma_distance', 'sigma_model']);

## Posterior fit

This plot shows posterior uncertainty in the continuous mechanistic success-probability relationship $p_{base}(x)$. It does not include the additional variation in newly observed successes.

In [ ]:
plot_rate(
    idata["posterior"]["p_base"],
    golf,
    "Posterior mechanistic mean",
    median_label="posterior median",
);

## Posterior predictive check

Now generate new outcomes at the same putting distances from the fitted model. This uses the same graphical grammar as the prior predictive check; the difference is that these predictions are conditioned on the observed data.

Because the likelihood is Normal on a proportion, posterior predictive draws are also not mathematically restricted to $[0,1]$.

In [ ]:
with model:
    pm.sample_posterior_predictive(
        idata,
        var_names=["rate"],
        extend_inferencedata=True,
        random_seed=RANDOM_SEED,
    )

In [ ]:
plot_rate(
    idata["posterior_predictive"]["rate"],
    golf,
    "Posterior predictive check",
    ylim=None,
    show_bounds=True,
);

## Model criticism

These residuals compare the observed rates with the posterior-median mechanistic relationship $p_{base}(x)$. They are a check for systematic structure in the mechanism, not a posterior predictive interval.

In [ ]:
plot_residuals(idata, golf, var_name="p_base", title="Residuals after allowing model discrepancy");

## Decision: predictively adequate, but conceptually unsatisfying

The fit is now quite good and the residuals have little obvious structure. This is an important workflow lesson: **a model can pass the current predictive checks and still be worth revising**. Here the discomfort is structural—the Normal approximation can assign probability outside $[0,1]$ and sidesteps the exact binomial observation process. **Next notebook:** keep the Binomial likelihood and put discrepancy on the logit scale.